In [ ]:
!pip install basicsr==1.4.2 --no-deps
!pip install realesrgan==0.3.0

# basicsr hatasını düzelt
dosya = '/usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py'
with open(dosya, 'r') as f:
    icerik = f.read()
icerik = icerik.replace(
    'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
    'from torchvision.transforms.functional import rgb_to_grayscale'
)
with open(dosya, 'w') as f:
    f.write(icerik)
print('Hazır!')

In [ ]:
!wget https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P /content/weights/

import torch
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
upsampler = RealESRGANer(
    scale=4,
    model_path='/content/weights/RealESRGAN_x4plus.pth',
    model=model,
    tile=400,
    tile_pad=10,
    pre_pad=0,
    half=True
)
print('Model hazır!')

In [ ]:
import cv2
import os

raw_klasor = '/content/drive/MyDrive/raw-890'
iyilestirilmis_klasor = '/content/drive/MyDrive/iyilestirilmis-890'
os.makedirs(iyilestirilmis_klasor, exist_ok=True)

dosyalar = os.listdir(raw_klasor)
toplam = len(dosyalar)

for i, dosya in enumerate(dosyalar):
    raw_yol = os.path.join(raw_klasor, dosya)
    cikti_yol = os.path.join(iyilestirilmis_klasor, dosya)

    # Zaten işlendiyse atla
    if os.path.exists(cikti_yol):
        continue

    img = cv2.imread(raw_yol, cv2.IMREAD_COLOR)
    if img is None:
        continue

    output, _ = upsampler.enhance(img, outscale=2)
    cv2.imwrite(cikti_yol, output)
    print(f'{i+1}/{toplam} → {dosya}')

print('Tüm görüntüler iyileştirildi!')

In [ ]:
import cv2
import numpy as np
import os
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

ref_klasor = '/content/drive/MyDrive/reference-890-extracted/reference-890'
iyilestirilmis_klasor = '/content/drive/MyDrive/iyilestirilmis-890'
raw_klasor = '/content/drive/MyDrive/raw-890'

psnr_raw_listesi = []
ssim_raw_listesi = []
psnr_iyi_listesi = []
ssim_iyi_listesi = []

for dosya in os.listdir(raw_klasor):
    raw_yol = os.path.join(raw_klasor, dosya)
    ref_yol = os.path.join(ref_klasor, dosya)
    iyi_yol = os.path.join(iyilestirilmis_klasor, dosya)

    if not os.path.exists(ref_yol) or not os.path.exists(iyi_yol):
        continue

    raw = cv2.imread(raw_yol)
    ref = cv2.imread(ref_yol)
    iyi = cv2.imread(iyi_yol)

    # Hepsini RAW boyutuna getir
    hedef_boyut = (raw.shape[1], raw.shape[0])
    ref_r = cv2.resize(ref, hedef_boyut)
    iyi_r = cv2.resize(iyi, hedef_boyut)  # İyileştirilmiş de küçültülüyor

    psnr_raw_listesi.append(psnr(ref_r, raw))
    ssim_raw_listesi.append(ssim(ref_r, raw, channel_axis=2))
    psnr_iyi_listesi.append(psnr(ref_r, iyi_r))
    ssim_iyi_listesi.append(ssim(ref_r, iyi_r, channel_axis=2))

print('========== SONUÇLAR ==========')
print(f'HAM GÖRÜNTÜ     → PSNR: {np.mean(psnr_raw_listesi):.2f} dB | SSIM: {np.mean(ssim_raw_listesi):.4f}')
print(f'İYİLEŞTİRİLMİŞ  → PSNR: {np.mean(psnr_iyi_listesi):.2f} dB | SSIM: {np.mean(ssim_iyi_listesi):.4f}')
fark_psnr = np.mean(psnr_iyi_listesi) - np.mean(psnr_raw_listesi)
fark_ssim = np.mean(ssim_iyi_listesi) - np.mean(ssim_raw_listesi)
print(f'FARK            → PSNR: {fark_psnr:+.2f} dB | SSIM: {fark_ssim:+.4f}')

outscale=1 → "Büyütme, sadece iyileştir"
→ Model daha az değişiklik yapıyor
→ Görsel fark az

outscale=4 → "4 kat büyüt"
→ Model çok daha fazla piksel dolduruyor
→ Bu sırada çok daha fazla iyileştirme yapıyor
→ Görsel fark belirgin!

PSNR yorumu: 16.58 → 16.29, çok küçük bir düşüş. Piksel seviyesinde model referanstan biraz uzaklaşmış ama fark minimal.SSIM yorumu: 0.7428 → 0.6579, bu daha ciddi. Yapısal benzerlik 0.085 düştü. Model görüntünün dokusunu, kenarlarını değiştirmiş — referansın beklediği yapıdan uzaklaşmış.Genel: Model görüntüyü işliyor ama sualtı renk bozulmasını düzeltmek yerine genel keskinleştirme yapıyor.

In [ ]:
import cv2
import os

raw_klasor = '/content/drive/MyDrive/raw-890'
iyi_klasor = '/content/drive/MyDrive/iyilestirilmis-890'

dosyalar = [f for f in os.listdir(iyi_klasor)][:5]

for dosya in dosyalar:
    raw = cv2.imread(os.path.join(raw_klasor, dosya))
    iyi = cv2.imread(os.path.join(iyi_klasor, dosya))

    if raw is None or iyi is None:
        continue

    kat = iyi.shape[0] / raw.shape[0]
    print(f'{dosya}:')
    print(f'  RAW: {raw.shape[1]}x{raw.shape[0]}')
    print(f'  İYİ: {iyi.shape[1]}x{iyi.shape[0]}')
    print(f'  Büyütme: {kat:.1f}x')
    print()

In [ ]:
import os

raw_klasor = '/content/drive/MyDrive/raw-890'
iyi_klasor = '/content/drive/MyDrive/iyilestirilmis-890'
ref_klasor = '/content/drive/MyDrive/reference-890-extracted/reference-890'

raw_dosyalar = set(os.listdir(raw_klasor))
iyi_dosyalar = set(os.listdir(iyi_klasor))
ref_dosyalar = set(os.listdir(ref_klasor))

# Üçünde de olan dosyalar
ortak = raw_dosyalar & iyi_dosyalar & ref_dosyalar
print(f'Üçünde de olan görüntü sayısı: {len(ortak)}')
print('Örnek:', list(ortak)[:5])